# Don't Make Fast Questions Feel Slow: Routing Queries to the Right Model

This notebook builds a customer support system that routes queries to different LLMs based on complexity using Elasticsearch Workflows. Mistral Small handles routing and simple FAQ answers. Claude Sonnet handles complex queries that need synthesis across multiple articles.

## Setup

In [ ]:
%pip install -qU elasticsearch python-dotenv requests datasets

In [ ]:
from dotenv import load_dotenv
import os
import requests
import json

load_dotenv()

ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
KIBANA_URL = os.getenv("KIBANA_URL")
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

## Connect to Elasticsearch

In [ ]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch(
    ELASTICSEARCH_URL, api_key=ELASTICSEARCH_API_KEY, request_timeout=30
)
es_client.info()

## Set up AI Connectors

We use two AI connectors: a custom Mistral Small connector and the Elastic Managed Claude Sonnet connector (already available). We only need to create the Mistral one.

In [ ]:
SMALL_LLM_CONNECTOR = "Mistral Small"

headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

# Create custom Mistral connector (OpenAI-compatible)
mistral_connector_payload = {
    "connector_type_id": ".gen-ai",
    "name": SMALL_LLM_CONNECTOR,
    "config": {
        "apiProvider": "Other",
        "apiUrl": "https://api.mistral.ai/v1/chat/completions",
        "defaultModel": "mistral-small-latest",
    },
    "secrets": {
        "apiKey": MISTRAL_API_KEY,
    },
}

# Let Kibana generate the UUID automatically
response = requests.post(
    f"{KIBANA_URL}/api/actions/connector",
    headers=headers,
    json=mistral_connector_payload,
)
result = response.json()
MISTRAL_CONNECTOR_ID = result.get("id")
print(f"Mistral connector created: {response.status_code}")
print(f"Connector ID: {MISTRAL_CONNECTOR_ID}")

## Load and index the dataset

We use the `e-commerce-customer-support-qa` dataset from Hugging Face. It contains 1,000 customer support interactions with categories, complexity levels, and product information. We index the Q&A pairs as our knowledge base.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("rjac/e-commerce-customer-support-qa", split="train")
print(f"Total entries: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
dataset[0]

In [ ]:
INDEX_NAME = "support-knowledge-base"

if es_client.indices.exists(index=INDEX_NAME):
    es_client.indices.delete(index=INDEX_NAME)

es_client.indices.create(
    index=INDEX_NAME,
    mappings={
        "properties": {
            "conversation": {
                "type": "text",
                "copy_to": "semantic_content",
            },
            "qa": {
                "type": "text",
                "copy_to": "semantic_content",
            },
            "issue_area": {"type": "keyword"},
            "issue_category": {"type": "keyword"},
            "issue_sub_category": {"type": "keyword"},
            "issue_category_sub_category": {"type": "keyword"},
            "customer_sentiment": {"type": "keyword"},
            "product_category": {"type": "keyword"},
            "product_sub_category": {"type": "keyword"},
            "issue_complexity": {"type": "keyword"},
            "agent_experience_level": {"type": "keyword"},
            "agent_experience_level_desc": {"type": "text"},
            "semantic_content": {
                "type": "semantic_text",
                "inference_id": ".jina-embeddings-v5-text-small",
            },
        }
    },
)
print(f"Created index: {INDEX_NAME}")

In [ ]:
from elasticsearch import helpers


def build_bulk_actions(dataset, index_name):
    for i, item in enumerate(dataset):
        yield {
            "_index": index_name,
            "_id": i,
            "_source": dict(item),
        }


success, failed = helpers.bulk(
    es_client,
    build_bulk_actions(dataset, INDEX_NAME),
    refresh=True,
)
print(f"Indexed {success} documents into '{INDEX_NAME}'")

In [ ]:
# Verify: check the distribution of complexity levels
for complexity in ["less", "medium", "high"]:
    count = es_client.count(
        index=INDEX_NAME,
        query={"term": {"issue_complexity": complexity}},
    )["count"]
    print(f"  {complexity}: {count} articles")

## The Workflow

The routing step only sees metadata from the search results (categories, complexity labels, scores) - not the full documents. This keeps classification cheap. The full article content is only loaded in the response step.

### Elasticsearch Workflow definition

The following YAML defines the workflow. Configure it directly in the Workflow UI (**Elasticsearch > Workflows > Create a New Workflow**).

```yaml
name: support_query_router
description: >
  Routes customer queries to the appropriate LLM based on complexity.
  Searches the KB, classifies using only result metadata (cheap),
  then routes to a small or large model depending on complexity.
enabled: true

inputs:
  - name: query
    type: string
    description: The customer support query
    required: true

consts:
  indexName: support-knowledge-base

triggers:
  - type: manual

steps:
  # Step 1: Search the knowledge base using semantic search
  - name: search_es
    type: elasticsearch.search
    with:
      index: "{{ consts.indexName }}"
      query:
        semantic:
          field: semantic_content
          query: "{{ inputs.query }}"
      size: 5

  # Step 2: Classify using only METADATA (Mistral Small cheap)
  - name: classify_query
    type: ai.prompt
    with:
      connectorId: Mistral Small
      prompt: >
        You are a support query classifier. Based on the customer query
        and the metadata of the top knowledge base hits below, decide
        how this query should be handled.

        Return ONLY a JSON object with:
        - "complexity": "simple" if the top hit clearly matches a single
          FAQ (high score, low-complexity category, single product area),
          or "complex" if the query spans multiple categories, the top
          hits have medium/high complexity labels, or the results are
          weakly matched.
        - "reasoning": one-line explanation.

        Customer query: {{ inputs.query }}

        Top 5 results (metadata only):
        1. score={{ steps.search_es.output.hits.hits[0]._score }}, category={{ steps.search_es.output.hits.hits[0]._source.issue_category_sub_category }}, complexity={{ steps.search_es.output.hits.hits[0]._source.issue_complexity }}, product={{ steps.search_es.output.hits.hits[0]._source.product_category }}
        2. score={{ steps.search_es.output.hits.hits[1]._score }}, category={{ steps.search_es.output.hits.hits[1]._source.issue_category_sub_category }}, complexity={{ steps.search_es.output.hits.hits[1]._source.issue_complexity }}, product={{ steps.search_es.output.hits.hits[1]._source.product_category }}
        3. score={{ steps.search_es.output.hits.hits[2]._score }}, category={{ steps.search_es.output.hits.hits[2]._source.issue_category_sub_category }}, complexity={{ steps.search_es.output.hits.hits[2]._source.issue_complexity }}, product={{ steps.search_es.output.hits.hits[2]._source.product_category }}
        4. score={{ steps.search_es.output.hits.hits[3]._score }}, category={{ steps.search_es.output.hits.hits[3]._source.issue_category_sub_category }}, complexity={{ steps.search_es.output.hits.hits[3]._source.issue_complexity }}, product={{ steps.search_es.output.hits.hits[3]._source.product_category }}
        5. score={{ steps.search_es.output.hits.hits[4]._score }}, category={{ steps.search_es.output.hits.hits[4]._source.issue_category_sub_category }}, complexity={{ steps.search_es.output.hits.hits[4]._source.issue_complexity }}, product={{ steps.search_es.output.hits.hits[4]._source.product_category }}

  # Step 3: Route based on complexity
  - name: route_by_complexity
    type: if
    condition: "${{ steps.classify_query.output.complexity == 'simple' }}"
    steps:
      - name: answer_from_faq
        type: ai.prompt
        with:
          connectorId: Mistral Small
          prompt: >
            You are a customer support agent. Answer the customer's question
            using ONLY the FAQ article below. Be concise, friendly, and
            include specific steps if applicable.

            Customer query: {{ inputs.query }}

            FAQ article:
            {{ steps.search_es.output.hits.hits[0]._source | json }}
    else:
      - name: synthesize_answer
        type: ai.prompt
        with:
          connectorId: Anthropic Claude Sonnet 4.6
          prompt: >
            You are a senior customer support specialist. The customer's query
            requires careful analysis across multiple knowledge base articles.

            Provide a detailed, empathetic response that:
            1. Addresses all aspects of the customer's question
            2. Cites specific articles from the knowledge base (reference them
               by their question/title)
            3. Provides clear resolution steps
            4. Notes if any part of the query isn't covered by the KB

            Customer query: {{ inputs.query }}

            Knowledge base articles:
            {{ steps.search_es.output.hits.hits | json }}
```

## Using the Workflow as a tool in Agent Builder

Workflows can be exposed as tools in Agent Builder, adding a conversational layer where users interact through a chat interface.

After creating the workflow in the Kibana UI, copy its ID and set it below.

In [ ]:
WORKFLOW_ID = "workflow-aaf77e41-37cf-48a8-973b-c853f71e4fae"  # Copy from Elastic UI after creating the workflow

In [ ]:
# Create the workflow tool
workflow_tool_payload = {
    "id": "run_support_query_router",
    "type": "workflow",
    "description": (
        "Routes a customer support query through the triage workflow. "
        "Searches the knowledge base, classifies query complexity, and "
        "generates a response using the appropriate model. Use this tool "
        "whenever a customer asks a support question."
    ),
    "tags": ["support", "triage", "workflow"],
    "configuration": {
        "workflow_id": WORKFLOW_ID,
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=workflow_tool_payload,
)
print(f"Workflow tool created: {response.status_code}")
print(response.json())

In [ ]:
# Create the agent with the workflow tool
agent_payload = {
    "id": "support-query-agent",
    "name": "Support Query Agent",
    "description": "Customer support agent that routes queries through a multi-model workflow.",
    "labels": ["support", "e-commerce"],
    "configuration": {
        "instructions": (
            "You are a customer support assistant for BrownBox, an e-commerce platform. "
            "When a customer asks a support question, use the `run_support_query_router` tool "
            "to process it. The tool will search the knowledge base, classify the query, "
            "and generate an appropriate response.\n\n"
            "Present the response to the customer in a friendly, professional tone."
        ),
        "tools": [{"tool_ids": ["run_support_query_router"]}],
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/agents",
    headers=headers,
    json=agent_payload,
)
print(f"Agent created: {response.status_code}")
print(response.json())

### Test the agent

Testing with a simple query (single FAQ match) and a complex query (multiple issues across categories).

In [ ]:
# Simple query - should match a single FAQ article
simple_converse = {
    "agent_id": "support-query-agent",
    "input": "How do I track my order?",
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/converse",
    headers=headers,
    json=simple_converse,
)
print("Simple query response:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# Complex query - requires synthesizing multiple articles
complex_converse = {
    "agent_id": "support-query-agent",
    "input": (
        "I ordered an OTG last week and it arrived damaged. I also noticed "
        "I was charged twice on my credit card. I want a replacement for the "
        "OTG and a refund for the duplicate charge. Also, my account shows "
        "the wrong delivery address - can you update it?"
    ),
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/converse",
    headers=headers,
    json=complex_converse,
)
print("Complex query response:")
print(json.dumps(response.json(), indent=2))

## Cleanup

In [ ]:
# Delete Agent Builder resources
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/agents/support-query-agent", headers=headers
)
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/tools/run_support_query_router", headers=headers
)
print("Deleted Agent Builder resources")

# Delete custom Mistral connector
requests.delete(
    f"{KIBANA_URL}/api/actions/connector/{MISTRAL_CONNECTOR_ID}", headers=headers
)
print(f"Deleted connector: {MISTRAL_CONNECTOR_ID}")

# Delete knowledge base index
es_client.indices.delete(index=INDEX_NAME)
print(f"Deleted index: {INDEX_NAME}")